In [ ]:
"""
Survival analysis between CNA clusters, using covariates. 
It runs Cox models and Restricted Mean Survival Time (RMST) analysis 
cluster vs rest, and outputs figure 3 of the manuscript.
"""

import pandas as pd
import numpy as np
from pathlib import Path

# Plotting and Survival packages
import matplotlib.pyplot as plt
import matplotlib as mpl
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import multivariate_logrank_test
from statsmodels.stats.multitest import fdrcorrection

# Rpy2 for RMST via survRM2
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

# Project-specific imports
from gbmcollaboration import CHUVDataCenter, UKERDataCenter, CRDataCenter
from gbmcollaboration import SURVIVAL_EXCLUSION
from gbmcollaboration.definitions import AVG_NUM_DAYS_PER_YEAR
from gbmcollaboration.utils.clinical_functions import plot_forest
from gbmcollaboration.utils.viz_survival import kmplot
from gbmcollaboration.viz import set_mpl_rcparams, savefig, pretty_forest_plot

set_mpl_rcparams()

# ---------------------------------------------------------------------------
# DATA LOADING FUNCTIONS
# ---------------------------------------------------------------------------
def load_clinical_data() -> pd.DataFrame:
    """Loads and concatenates tabular clinical data from the 3 main centers."""
    merged_df = []
    centers =[CHUVDataCenter, UKERDataCenter, CRDataCenter]

    for center in centers:
        center_data_dict = center.load_tabular(
            exclude_idh1_mut=True,
            exclude_patient_ids=True,
            exclude_relapse=True
        )
        df = center_data_dict['clinical']['processed gbm clinical']

        if center == CRDataCenter:
            df = df.drop(index=SURVIVAL_EXCLUSION)

        print(f"\n--- Loading the clinical data from {center.name} ---")
        print(df['Sample collection chronology'].value_counts())
        print(f"Found {len(df)} primary cases")
        merged_df.append(df)

    df_concat = pd.concat(merged_df, ignore_index=False)
    print("\nDataframes from the three centers have been merged.")
    return df_concat


def load_and_merge_cluster_data(clinical_df: pd.DataFrame, endpoint: str) -> tuple[pd.DataFrame, str]:
    """Loads clustering annotations and merges them with the clinical dataframe."""
    df_wes = pd.read_csv(CLUSTER_CSV)
    df_wes.set_index("orig.ident", inplace=True)

    cluster_col = "cnv_leiden"

    # Map categorical covariates
    mapping = {
        "MGMT promoter methylation": {"Unmethylated": 0, "Methylated": 1},
        "Administrative gender": {"Male": 0, "Female": 1},
        "Site code": {"CH": 0, "ER": 1, "CR": 2}
    }

    df_surv_cov = clinical_df[[endpoint] + COVARIATE_COLS].copy()
    df_surv_cov = df_surv_cov.replace(mapping)

    df_cluster = df_wes[[cluster_col]].merge(
        df_surv_cov, how="inner", left_index=True, right_index=True
    ).copy()

    df_cluster[cluster_col] = df_cluster[cluster_col].astype(int)

    # Build time/event in standard lifelines format
    df_cluster[f"{endpoint}.time"] = df_cluster[endpoint].abs()
    df_cluster[f"{endpoint}.event"] = (df_cluster[endpoint] > 0).astype(int)
    df_cluster[f'{endpoint}.time.days'] = df_cluster[f'{endpoint}.time'] * AVG_NUM_DAYS_PER_YEAR

    print(f"\nMerged df shape: {df_cluster.shape}")
    print("Cluster counts:\n", df_cluster[cluster_col].value_counts().sort_index())
    return df_cluster, cluster_col


# ---------------------------------------------------------------------------
# ANALYSIS FUNCTIONS
# ---------------------------------------------------------------------------
def run_adjusted_cox_cluster_vs_rest(df_cluster: pd.DataFrame, cluster_col: str, clusters: list, time_col: str, event_col: str) -> pd.DataFrame:
    """Iterates through all clusters and runs a CoxPH fit (Cluster vs Rest)."""
    df_cluster_vs_rest_rows =[]

    for cl in clusters:
        tmp = df_cluster.copy()
        tmp["is_cluster"] = (tmp[cluster_col] == cl).astype(int)

        use_cols = [time_col, event_col, "is_cluster"] + COVARIATE_COLS
        tmp = tmp[use_cols].dropna()

        n_in = int(tmp["is_cluster"].sum())
        n_out = int((1 - tmp["is_cluster"]).sum())

        cph = CoxPHFitter()
        cph.fit(tmp, duration_col=time_col, event_col=event_col, show_progress=False)

        # Extract effect for cluster
        s = cph.summary.loc["is_cluster"]
        df_cluster_vs_rest_rows.append({
            "cluster": cl, "n_in": n_in, "n_out": n_out,
            "HR": float(s["exp(coef)"]),
            "CI95_rest": float(s["exp(coef) lower 95%"]),
            "CI95_cluster": float(s["exp(coef) upper 95%"]),
            "p": float(s["p"])
        })

        # Forest plot visualization
        df_sum = cph.summary.reset_index().rename(columns={"index": "covariate"})
        _, df_sum["fdr"] = fdrcorrection(df_sum["p"].values)

        plot_forest(df_all=df_sum, distances=[0.45, 0.15, 0.35], figsize=(10, 3.8), outcome="OS", significance="FDR")
        plt.title(f"Adjusted Cox: cluster {cl} vs all others")
        plt.show()

    df_cox_res = pd.DataFrame(df_cluster_vs_rest_rows).sort_values("p")
    print("\nAdjusted Cox (cluster vs rest) summary table:")
    print(df_cox_res)
    return df_cox_res


def run_rmst_and_km_cluster_vs_rest(df_cluster: pd.DataFrame, cluster_col: str, cl: int, time_col: str, event_col: str):
    """Calculates RMST stats in R and plots KM curves for an individual cluster."""
    tmp = df_cluster.copy()
    tmp["is_cluster"] = (tmp[cluster_col] == cl).astype(int)

    use_cols = [time_col, event_col, "is_cluster"] + COVARIATE_COLS
    tmp = tmp[use_cols].dropna()

    n_rest, n_cluster = tmp['is_cluster'].value_counts().sort_index()

    # Prepping columns to be legally valid names for R
    r_cov_names =[c.replace(" ", "_").replace("(", "").replace(")", "") for c in COVARIATE_COLS]
    r_df = tmp.copy()
    r_df.columns = ["time", "event", "arm"] + r_cov_names

    adjust = len(COVARIATE_COLS) > 0

    try:
        with localconverter(ro.default_converter + pandas2ri.converter):
            ro.globalenv["surv_data"] = ro.conversion.py2rpy(r_df)
        ro.globalenv["tau"] = TAU_YEARS

        ro.r("library(survRM2)")
        ro.r("rmst_unadj <- rmst2(time=surv_data$time, status=surv_data$event, arm=surv_data$arm, tau=tau)")

        unadj = np.array(ro.r("rmst_unadj$unadjusted.result[1,]"))
        rmst_cluster = float(ro.r("rmst_unadj$RMST.arm1$rmst['Est.']")[0])
        rmst_rest = float(ro.r("rmst_unadj$RMST.arm0$rmst['Est.']")[0])
        diff_unadj, ci_low_unadj, ci_high_unadj, p_unadj = unadj

        print(f"\n{'='*60}\nUNADJUSTED RMST (Cluster {cl} vs Rest)\n{'='*60}")
        print(f"  RMST Cluster {cl}: {rmst_cluster:.3f} yr ({rmst_cluster*12:.1f} months)")
        print(f"  RMST Rest:       {rmst_rest:.3f} yr ({rmst_rest*12:.1f} months)")
        print(f"  Difference:      {diff_unadj:+.3f} yr ({diff_unadj*12:+.1f} months)")
        print(f"  p-value:         {p_unadj:.4f}")

        if adjust:
            cov_r_str = ", ".join([f'"{c}"' for c in r_cov_names])
            ro.r(f"""
            cov_mat <- as.matrix(surv_data[, c({cov_r_str})])
            rmst_adj <- rmst2(time=surv_data$time, status=surv_data$event, arm=surv_data$arm, tau=tau, covariates=cov_mat)
            """)
            adj = np.array(ro.r("rmst_adj$adjusted.result[1,]"))
            diff_adj, ci_low_adj, ci_high_adj, p_adj = adj


            cov_label = " + ".join(COVARIATE_COLS)

            print(f"\nADJUSTED RMST ({cov_label}, n={len(tmp)})")
            print(f"  Adj difference: {diff_adj:+.3f} yr ({diff_adj*12:+.1f} months)")
            print(f"  Adj p-value:    {p_adj:.4f}")
        else:
            diff_adj, ci_low_adj, ci_high_adj, p_adj, cov_label = diff_unadj, ci_low_unadj, ci_high_unadj, p_unadj, "none"

    except Exception as e:
        print(f"RMST failed for Cluster {cl}: {e}")
        return None

    # --- Plot Kaplan-Meier via Python's lifelines ---
    fig, ax = plt.subplots(figsize=(8, 6))
    km_colors = {1: COLOR_CLUSTER, 0: COLOR_REST}
    km_labels = {1: f"Cluster {cl}", 0: 'Rest'}

    for grp in [1, 0]:
        mask = tmp["is_cluster"] == grp
        kmf = KaplanMeierFitter()
        kmf.fit(tmp.loc[mask, time_col], tmp.loc[mask, event_col], label=km_labels[grp])
        kmf.plot_survival_function(ax=ax, color=km_colors[grp], linewidth=2.5, ci_show=True, ci_alpha=0.15)

        censored_times = tmp.loc[mask & (tmp[event_col] == 0), time_col]
        for t in censored_times:
            ax.plot(t, kmf.predict(t), "|", color=km_colors[grp], markersize=8, markeredgewidth=1.5)

    ax.axvline(TAU_YEARS, color="gray", linestyle="--", linewidth=1.2, alpha=0.7, label=f"τ={TAU_YEARS}yr")

    annot = (
        f"RMST (τ={TAU_YEARS}yr):\n"
        f"  Cluster {cl}: {rmst_cluster:.2f} yrs\n"
        f"  Rest: {rmst_rest:.2f} yrs\n"
        f"  Δ: {diff_adj*12:+.1f} months\n"
        f"  p(adj) = {p_adj:.4f}"
    )

    ax.text(0.97, 0.97, annot, transform=ax.transAxes, fontsize=12, va="top", ha="right",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.8, edgecolor="gray"))

    ax.set_title(f"{ENDPOINT} Survival: Cluster {cl} (n={n_cluster}) vs Rest (n={n_rest})", fontsize=14, fontweight="bold")
    ax.set_xlabel("Time (years)", fontsize=12)
    ax.set_ylabel("Survival probability", fontsize=12)
    ax.set_ylim(0, 1.05)
    ax.set_xlim(0, None)
    ax.legend(loc="lower left", frameon=False, fontsize=12)
    ax.spines[["top", "right"]].set_visible(False)
    
    plt.tight_layout()

    # Save output
    save_path = FIGURES_DIR / f"figure3/Cluster_{cl}_vs_Rest_{ENDPOINT}_RMST_KM.pdf"
    save_path.parent.mkdir(parents=True, exist_ok=True)
    savefig(df=tmp, out_file=save_path)
    plt.show()

    return {
        "cluster": cl, "n_cluster": n_cluster, "n_rest": n_rest, "tau_yr": TAU_YEARS,
        "rmst_cluster_yr": rmst_cluster, "rmst_rest_yr": rmst_rest,
        "diff_months_unadj": diff_unadj * 12, "ci_months_low_unadj": ci_low_unadj * 12, "ci_months_high_unadj": ci_high_unadj * 12, "p_unadj": p_unadj,
        "diff_months_adj": diff_adj * 12, "ci_months_low_adj": ci_low_adj * 12, "ci_months_high_adj": ci_high_adj * 12, "p_adj": p_adj,
    }



def run_adjusted_categorical_cox(df_cluster: pd.DataFrame, cluster_col: str, clusters: list, time_col: str, event_col: str):
    """Computes fixed categorical Cox logic using largest set as a reference cluster."""
    ref_cluster = 0 if 0 in clusters else int(df_cluster[cluster_col].value_counts().idxmax())
    print(f"\nReference cluster for categorical Cox: {ref_cluster}")

    df_cat = df_cluster[[cluster_col, time_col, event_col] + COVARIATE_COLS].dropna().copy()
    df_cat[cluster_col] = pd.Categorical(df_cat[cluster_col])

    new_cats = [ref_cluster] + [c for c in df_cat[cluster_col].cat.categories if c != ref_cluster]
    df_cat[cluster_col] = df_cat[cluster_col].cat.reorder_categories(new_cats)
    df_cat_dum = pd.get_dummies(df_cat, columns=[cluster_col], drop_first=True)

    cph_cat = CoxPHFitter()
    cph_cat.fit(df_cat_dum, duration_col=time_col, event_col=event_col, show_progress=False)

    df_sum_cat = cph_cat.summary.reset_index().rename(columns={"index": "covariate"})
    _, df_sum_cat["fdr"] = fdrcorrection(df_sum_cat["p"].values)

    # Dynamic Covariate Mapping
    covariate_map = {col: col for col in COVARIATE_COLS}
    for c in clusters:
        if c != ref_cluster:
            covariate_map[f"{cluster_col}_{c}"] = f"CNA cluster {c}"

    df_sum_cat["covariate"] = df_sum_cat["covariate"].map(covariate_map).fillna(df_sum_cat["covariate"])

    mpl.rcParams.update({"font.size": 20, "axes.titlesize": 20, "axes.labelsize": 20, "legend.fontsize": 20})

    plt = plot_forest(df_all=df_sum_cat, distances=[0.60, 0.10, 0.50], figsize=(10, 8), outcome=time_col, significance="FDR")
    plt.title(f"Adjusted categorical Cox (reference cluster : {ref_cluster})")

    out_file = FIGURES_DIR / "figure3/fig3d.pdf"
    out_file.parent.mkdir(parents=True, exist_ok=True)
    savefig(df=df_sum_cat, out_file=out_file)
    plt.show()


def plot_global_km_curves(df_cluster: pd.DataFrame, cluster_col: str, time_col: str, event_col: str):
    """Outputs Figure 3c KM logic plotting multivariate lines simultaneously"""
    df_km = df_cluster[[cluster_col, time_col, event_col]].copy()
    df_km[cluster_col] = df_km[cluster_col].astype(str)
    hue_order = sorted(df_km[cluster_col].unique(), key=lambda x: int(x))

    # Single KM
    fig, ax = plt.subplots(1, 1, figsize=(9, 7))
    kmplot(data=df_km, x=time_col, hue=cluster_col, hue_order=hue_order,
           convert_days_to_months=True, add_hazard_ratio=False, add_logrank_pvalue=False,
           at_risk_counts=False, ax=ax, title="Overall survival by WES CNV Leiden cluster")
    plt.tight_layout()
    savefig(df=df_km, out_file=FIGURES_DIR / "figure3/fig3c_km.pdf")
    plt.show()

    # KM + Log Rank Text
    fig, ax = plt.subplots(1, 1, figsize=(9, 7))
    kmplot(data=df_km, x=time_col, hue=cluster_col, hue_order=hue_order,
           convert_days_to_months=True, add_hazard_ratio=False, add_logrank_pvalue=False,
           at_risk_counts=False, ax=ax, title="Overall survival by WES CNV Leiden cluster")

    df_lr = df_km.dropna(subset=[time_col, cluster_col]).copy()
    res = multivariate_logrank_test(event_durations=df_lr[time_col], groups=df_lr[cluster_col], event_observed=df_lr[event_col])

    p = float(res.p_value)
    p_txt = "Global log-rank: p<0.001" if p < 0.001 else f"Global log-rank: p={p:.3g}"
    ax.text(0.02, 0.02, p_txt, transform=ax.transAxes, fontsize=12, va="bottom",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.8, edgecolor="gray"))
    
    plt.tight_layout()
    savefig(df=df_km, out_file=FIGURES_DIR / "figure3/fig3c_km_mulvariate_test.pdf")
    plt.show()

In [ ]:
# ===== USER CONFIG =====
TAU_YEARS = 3
ENDPOINT = "OS"
CLINICAL_ID_COL = "Subject Id"
COVARIATE_COLS = ["Age (years)", "MGMT promoter methylation", "Administrative gender", "Site code"]

# Plot aesthetics
figsize = (9, 6)
title_fontsize = 8
axis_label_fontsize = 12
tick_fontsize = 12
legend_fontsize = 20
annot_fontsize = 12
km_linewidth = 2
censor_markersize = 6
censor_markeredgewidth = 1.5

CLUSTER_CSV = "/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/results/WES/infercnv/leiden_cluster_wes_baseline_samples.csv"

FIGURES_DIR = Path('/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/figures/')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
COLOR_CLUSTER = "#fe7a02"    # orange
COLOR_REST = "#007ebc"     # blue


# Endpoint mapping
if ENDPOINT.upper() == "OS":
    time_col, event_col = "OS.time", "OS.event"
    time_label = "Overall Survival"
elif ENDPOINT.upper() == "PFS":
    time_col, event_col = "PFS.time", "PFS.event"
    time_label = "Progression-Free Survival"
else:
    raise ValueError("ENDPOINT must be 'OS' or 'PFS'")

In [ ]:
# df_clinical = load_clinical_data()
df_cluster, cluster_col = load_and_merge_cluster_data(df_clinical,endpoint=ENDPOINT)

clusters = sorted(df_cluster[cluster_col].unique())

print("\n" + "#" * 60)
print("1. Survival Analysis (Cluster vs Rest)")
print("#" * 60)

# 1A: Loop over basic adjusted Cox tests
run_adjusted_cox_cluster_vs_rest(df_cluster, cluster_col, clusters, time_col=time_col, event_col=event_col)

# 1B: Loop over RMST and generate Cluster vs Rest individual Plots 
rmst_results = []
for cl in clusters:
    print(f"\n{'#'*60}\nRunning RMST for Cluster {cl} vs Rest\n{'#'*60}")
    res = run_rmst_and_km_cluster_vs_rest(df_cluster, cluster_col, cl, time_col=time_col, event_col=event_col)
    if res:
        rmst_results.append(res)

# Save cumulative RMST summary
if rmst_results:
    df_rmst_summary = pd.DataFrame(rmst_results)
    summary_path = FIGURES_DIR / "figure3/cluster_vs_rest_rmst_summary.csv"
    df_rmst_summary.to_csv(summary_path, index=False)
    print(f"\nRMST summary table saved to: {summary_path}")
    print(df_rmst_summary.to_string(index=False))

print("\n" + "#" * 60)
print("2. Categorical Cox Analysis (Fixed Reference)")
print("#" * 60)

# 2A: Adjusted categorical Cox (Figure 3d)
run_adjusted_categorical_cox(df_cluster, cluster_col, clusters, time_col=time_col, event_col=event_col)

print("\n" + "#" * 60)
print("3. Global Survival Plots (Figure 3c)")
print("#" * 60)

# 3A: Global Kaplan-Meier and Log-rank test
plot_global_km_curves(df_cluster, cluster_col, time_col=time_col, event_col=event_col)

print("\nPipeline execution complete.")

# Forest plot paper fig

In [ ]:
def pretty_forest_plot(
    df: pd.DataFrame,
    
    # --- Column Mappings (FLEXIBLE INPUTS) ---
    col_label: str = "cluster",
    col_n_comp: str = "n_cluster",
    col_n_ref: str = "n_rest",
    col_diff: str = "diff_months_adj",
    col_lo: str = "ci_months_low_adj",
    col_hi: str = "ci_months_high_adj",
    col_p: str = "p_adj",
    
    # --- Scaling (Useful if diff is in months but CIs are in years) ---
    diff_multiplier: float = 1.0,
    ci_multiplier: float = 1.0,
    
    # --- Output & Titling ---
    save_path: str = None,
    title: str = "Adjusted RMST difference by cluster",
    xlabel: str = "RMST difference (months)  |  cluster vs rest",
    tau: int = None,
    
    # --- Figure Dimensions & Resolution ---
    figsize: tuple = (15, 5),
    dpi: int = 300,
    subplots_adj_left: float = 0.38,
    subplots_adj_right: float = 0.72,
    
    # --- Fonts ---
    fs_hdr: float = 8.5,
    fs_row: float = 8.5,
    fs_axis: float = 8.5,
    fs_title: float = 9.5,
    
    # --- Marker & Line Styles ---
    marker: str = "D",
    marker_sz: float = 5,
    cap_sz: float = 3,
    lw: float = 1.3,
    elw: float = 1.1,
    
    # --- Colors ---
    col_sig: str = "#C0392B",
    col_ns: str = "#2C3E50",
    col_n_text: str = "gray",
    col_vline: str = "#7F8C8D",
    grid_alpha: float = 0.4,
    
    # --- Column Positioning (transAxes: 0 is left axis edge, 1 is right axis edge) ---
    x_label: float = -0.25,  # covariate label (right-aligned)
    x_n: float     = -0.02,  # n column        (right-aligned)
    x_est: float   = 1.02,   # estimate + CI   (left-aligned)
    x_p: float     = 1.4    # p-value         (left-aligned)
):
    # Helper to format p-values
    def fmt_p(p):
        if pd.isna(p): return "—"
        if p < 0.0001: return f"{p:.2e}"
        return f"{p:.4f}"

    n_rows = len(df)
    
    # Append tau to titles if provided
    if tau is not None:
        title += f" (τ={int(tau)} years)"
        xlabel = xlabel.replace("months)", f"months, τ={int(tau)} years)")

    # 1. Setup Plot
    fig, ax = plt.subplots(figsize=figsize)
    y_pos = np.arange(n_rows - 1, -1, -1, dtype=float)

    # 2. Alternating row shading
    for i in range(n_rows):
        if i % 2 == 0:
            ax.axhspan(y_pos[i] - 0.45, y_pos[i] + 0.45, color="whitesmoke", zorder=0)

    # Track min/max for x-axis scaling
    all_lo, all_hi = [], []

    # 3. Draw Error bars
    for i, (_, row) in enumerate(df.iterrows()):
        y = y_pos[i]
        
        # Extract and scale values
        p_val = row[col_p]
        is_sig = False if pd.isna(p_val) else p_val < 0.05
        c_color = col_sig if is_sig else col_ns
        
        val_diff = row[col_diff] * diff_multiplier
        val_lo   = row[col_lo] * ci_multiplier
        val_hi   = row[col_hi] * ci_multiplier
        
        if pd.notna(val_diff) and pd.notna(val_lo) and pd.notna(val_hi):
            ax.errorbar(
                x=val_diff, y=y,
                xerr=[[val_diff - val_lo], [val_hi - val_diff]],
                fmt=marker, color=c_color, markersize=marker_sz,
                capsize=cap_sz, linewidth=lw, elinewidth=elw, zorder=3,
            )
            all_lo.append(val_lo)
            all_hi.append(val_hi)

    # 4. Reference line & Axes styling
    ax.axvline(x=0, color=col_vline, linewidth=0.9, linestyle="--", zorder=2)

    # Dynamic x-axis limits based on SCALED values
    x_min_data = min(all_lo) if all_lo else -5
    x_max_data = max(all_hi) if all_hi else 5
    x_pad = max((x_max_data - x_min_data) * 0.08, 1.0)
    
    ax.set_xlim(x_min_data - x_pad, x_max_data + x_pad)
    ax.set_ylim(-0.7, n_rows - 0.3)
    ax.set_yticks([]) # Hide y-ticks
    
    ax.set_xlabel(xlabel, fontsize=fs_axis)
    ax.set_title(title + "\n", fontsize=fs_title, pad=8)
    
    for sp in ["top", "right", "left"]:
        ax.spines[sp].set_visible(False)
    ax.spines["bottom"].set_linewidth(0.6)
    ax.tick_params(axis="x", labelsize=fs_axis, width=0.6, length=3)
    ax.grid(axis="x", linewidth=0.4, alpha=grid_alpha, zorder=0)

    # 5. Text Annotations
    trans = ax.transAxes
    y_lo_ax, y_hi_ax = ax.get_ylim()
    y_to_frac = lambda y_data: (y_data - y_lo_ax) / (y_hi_ax - y_lo_ax)

    # Headers
    header_frac = y_to_frac(n_rows - 0.05)
    ax.text(x_label, header_frac, "Cluster", transform=trans, fontsize=fs_hdr, fontweight="bold", ha="right", va="bottom")
    ax.text(x_n,     header_frac, "n (clust/rest)", transform=trans, fontsize=fs_hdr, fontweight="bold", ha="right", va="bottom")
    ax.text(x_est,   header_frac, "Estimate [95% CI] (months)", transform=trans, fontsize=fs_hdr, fontweight="bold", ha="left", va="bottom")
    ax.text(x_p,     header_frac, "adj p", transform=trans, fontsize=fs_hdr, fontweight="bold", ha="left", va="bottom")

    # Rows
    for i, (_, row) in enumerate(df.iterrows()):
        yf = y_to_frac(y_pos[i])
        
        p_val = row[col_p]
        is_sig = False if pd.isna(p_val) else p_val < 0.05
        c_color = col_sig if is_sig else col_ns

        # Covariate label (cast to string in case it's an integer like 0, 1, 2)
        lbl_str = row[col_label]
        ax.text(x_label, yf, lbl_str, transform=trans, fontsize=fs_row, ha="right", va="center", color=col_ns)

        # N
        n_str = f"{int(row[col_n_comp])}/{int(row[col_n_ref])}" if pd.notna(row[col_n_comp]) else "—/—"
        ax.text(x_n, yf, n_str, transform=trans, fontsize=fs_row, ha="right", va="center", color=col_n_text)

        # Estimate and P-value
        val_diff = row[col_diff] * diff_multiplier
        val_lo   = row[col_lo] * ci_multiplier
        val_hi   = row[col_hi] * ci_multiplier

        if pd.notna(val_diff):
            est_str = f"{val_diff:+.1f} [{val_lo:+.1f}, {val_hi:+.1f}]"
            p_str = fmt_p(p_val) + (" *" if is_sig else "")
        else:
            est_str, p_str = "—", "—"

        ax.text(x_est, yf, est_str, transform=trans, fontsize=fs_row, ha="left", va="center", color=c_color)
        ax.text(x_p, yf, p_str, transform=trans, fontsize=fs_row, ha="left", va="center", color=c_color)

    # 6. Final Adjustments & Save
    fig.subplots_adjust(left=subplots_adj_left, right=subplots_adj_right)

    if save_path:
        savefig(df=df, out_file=save_path)
    
    plt.show()

df_rmst_summary['cluster'] = df_rmst_summary['cluster'].astype(str)
pretty_forest_plot(df_rmst_summary,save_path=FIGURES_DIR / "figure3/fig3e_rmst_forest_plot.pdf")